In [2]:
import os
from pathlib import Path

project_root = Path.cwd().parent
os.chdir(project_root)
print("CWD:", project_root)

from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from src.rag_pipeline import RagPipeline

CWD: /home/gusevsaint/Workspace/study/practice/mental-helper


/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
    filename="qwen2.5-0.5b-instruct-q4_k_m.gguf",
)
rag = RagPipeline()
llm = Llama(model_path=model_path, n_ctx=2048, n_threads=4, verbose=False)

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [ ]:
def generate_answer(query, top_k=4, max_tokens=256):
    retrieved = rag.retrieve(query, top_k=top_k)
    context = "\n\n".join(
        f"[{i}] Q: {r['question']}\nA: {r['answer']}"
        for i, r in enumerate(retrieved, 1)
    )[:2500]

    prompt = f"""<|im_start|>system
You are a mental health assistant. Provide 4-6 clear, actionable points. Be specific and encouraging.<|im_end|>
<|im_start|>user
Context:
{context}

Question: {query}<|im_end|>
<|im_start|>assistant
"""

    output = llm(
        prompt, max_tokens=max_tokens, temperature=0.7, top_p=0.9, stop=["<|im_end|>"]
    )
    answer = output["choices"][0]["text"].strip()

    return {"answer": answer, "retrieved": retrieved}


def generate_baseline(query, max_tokens=256):
    prompt = f"""<|im_start|>system
You are a mental health assistant. Provide 4-6 clear, actionable points. Be specific and encouraging.<|im_end|>
<|im_start|>user
Question: {query}<|im_end|>
<|im_start|>assistant
"""

    output = llm(
        prompt, max_tokens=max_tokens, temperature=0.7, top_p=0.9, stop=["<|im_end|>"]
    )
    return output["choices"][0]["text"].strip()


# Тест
resp_rag = generate_answer("How to cope with stress?", top_k=4)
resp_base = generate_baseline("How to cope with stress?")

print("With RAG:\n", resp_rag["answer"], "\n")
print("Sources:")
for r in resp_rag["retrieved"]:
    print(f"  - {r['question']} (score: {r['score']:.3f})")

print("\nWithout RAG:\n", resp_base)

With RAG:
 Managing stress effectively is a powerful skill. Here are 4 actionable steps to help you manage stress:

1. **Identify Triggers**: Recognize what triggers your stress. Knowing what causes stress helps you anticipate and prepare for stressful situations. This is crucial for preparing your body and mind to handle stress.

2. **Practice Deep Breathing**: When you feel stressed, take slow, deep breaths. Inhale deeply for a count of four, hold for four, and then exhale for four. This simple technique can help calm your nervous system and reduce anxiety.

3. **Stay Active**: Engaging in regular physical activity can be a great stress reliever. Whether it's going for a walk, doing yoga, or any other form of exercise you enjoy, it releases endorphins that can boost your mood.

4. **Practice Mindfulness or Meditation**: Mindfulness techniques or meditation can help you stay in the present moment and reduce anxiety about the past or future. Spend time in the present by focusing on you